[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/engineerinvestor/lifecycle-allocation/blob/main/examples/notebooks/risky_human_capital.ipynb)

# Risky Human Capital: How Your Job Affects Your Portfolio

The standard lifecycle model treats all human capital as **bond-like**: stable, predictable cash flows that diversify a stock portfolio. That works well for government employees and tenured professors, but it breaks down for anyone whose income is tied to the stock market.

This notebook introduces the **human capital beta** feature, which lets you specify how much of your future earnings behave like equity vs. bonds. A higher beta means your income is more correlated with the stock market, which *reduces* the diversification benefit of human capital and *lowers* the recommended equity allocation.

We will walk through:
1. Why the bond-like assumption fails for many workers
2. How beta adjusts the allocation formula
3. Industry-level calibration using built-in beta estimates
4. Sensitivity analysis and case studies
5. Practical guidance for estimating your own beta

> **Disclaimer:** This is for education and research purposes only. It is not investment advice.

In [ ]:
# Install (run this cell on Colab; skip locally if already installed)
import sys

if "google.colab" in sys.modules:
    !pip install -q git+https://github.com/engineerinvestor/lifecycle-allocation.git

In [ ]:
import matplotlib.pyplot as plt

from lifecycle_allocation import (
    INDUSTRY_BETAS,
    HumanCapitalSpec,
    InvestorProfile,
    MarketAssumptions,
    alpha_star,
    recommended_stock_share,
    suggested_beta,
)

---
## The Problem: Not All Human Capital Is Bond-Like

Consider a software engineer at Google earning \$180k in base salary plus \$2M in unvested RSUs. Their total compensation is heavily tied to Google's stock price, which moves with the broader market. If the market crashes, their RSU value drops *and* their job security weakens (tech layoffs correlate with downturns).

The standard model would look at this person's \$2M+ in human capital and say: "You have a huge bond-like asset, so put everything in stocks." But that is wrong. Their human capital already *is* stock-like. Adding more equity to their portfolio concentrates risk rather than diversifying it.

The fix is simple: decompose human capital into a **bond-like** portion and an **equity-like** portion using a parameter called **beta**.

- `beta = 0`: All human capital is bond-like (government worker, tenured professor)
- `beta = 0.6`: 60% equity-like (tech worker with RSUs)
- `beta = 1.0`: Fully equity-like (startup founder paid entirely in equity)

The adjusted allocation formula uses only the bond-like fraction of H:

$$\alpha_{recommended} = \alpha^* \times \left(1 + \frac{(1 - \beta) \cdot H}{W}\right)$$

Higher beta shrinks the effective H/W ratio, producing a lower equity allocation.

---
## Standard vs. Beta-Adjusted Allocation

Let's compare the standard model (beta=0) with a beta-adjusted model (beta=0.6) for a 30-year-old tech worker.

In [ ]:
market = MarketAssumptions(mu=0.05, r=0.02, sigma=0.18)

# Standard model: beta = 0 (all HC is bond-like)
profile_standard = InvestorProfile(
    age=30,
    retirement_age=67,
    investable_wealth=100_000,
    after_tax_income=150_000,
    risk_tolerance=6,
    human_capital_model=HumanCapitalSpec(beta=0.0),
)

# Beta-adjusted model: beta = 0.6 (tech worker with RSUs)
profile_tech = InvestorProfile(
    age=30,
    retirement_age=67,
    investable_wealth=100_000,
    after_tax_income=150_000,
    risk_tolerance=6,
    human_capital_model=HumanCapitalSpec(beta=0.6),
)

r_standard = recommended_stock_share(profile_standard, market)
r_tech = recommended_stock_share(profile_tech, market)

hdr = f"{'Metric':<25} {'Standard':>15} {'Tech (b=0.6)':>15}"
print(hdr)
print("-" * len(hdr))
h_std = r_standard.human_capital
h_tch = r_tech.human_capital
c_std = r_standard.components
c_tch = r_tech.components
print(f"{'Human capital (H)':<25} ${h_std:>12,.0f}  ${h_tch:>12,.0f}")
hw_std = h_std / 100_000
hw_tch = h_tch / 100_000
print(f"{'H/W ratio':<25} {hw_std:>14.1f}x {hw_tch:>14.1f}x")
hb_s = c_std["human_capital_bond_like"]
hb_t = c_tch["human_capital_bond_like"]
print(f"{'Bond-like H':<25} ${hb_s:>12,.0f}  ${hb_t:>12,.0f}")
he_s = c_std["human_capital_equity_like"]
he_t = c_tch["human_capital_equity_like"]
print(f"{'Equity-like H':<25} ${he_s:>12,.0f}  ${he_t:>12,.0f}")
a_std = r_standard.alpha_unconstrained
a_tch = r_tech.alpha_unconstrained
print(f"{'Unconstrained alloc':<25} {a_std:>14.1%}  {a_tch:>14.1%}")
r_std = r_standard.alpha_recommended
r_tch = r_tech.alpha_recommended
print(f"{'Recommended alloc':<25} {r_std:>14.1%}  {r_tch:>14.1%}")

---
## Industry Comparison

The library ships with calibrated beta values for common industries. Let's see how the allocation changes across six representative industries.

In [ ]:
industries = [
    "government",
    "healthcare",
    "tech_salaried",
    "tech_with_rsus",
    "tech_startup",
    "startup_equity_heavy",
]

industry_results = []

print(f"{'Industry':<25} {'Beta':>6} {'Allocation':>12}")
print("-" * 45)

for ind in industries:
    beta = suggested_beta(ind)
    p = InvestorProfile(
        age=30,
        retirement_age=67,
        investable_wealth=100_000,
        after_tax_income=150_000,
        risk_tolerance=6,
        human_capital_model=HumanCapitalSpec(beta=beta, industry=ind),
    )
    r = recommended_stock_share(p, market)
    industry_results.append((ind, beta, r.alpha_recommended))
    print(f"{ind:<25} {beta:>5.2f}  {r.alpha_recommended:>11.1%}")

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(9, 5))
labels = [r[0].replace("_", " ").title() for r in industry_results]
allocs = [r[2] for r in industry_results]
betas = [r[1] for r in industry_results]
colors = [plt.cm.RdYlGn_r(b) for b in betas]

bars = ax.barh(labels, allocs, color=colors, edgecolor="white", linewidth=0.5)
for bar, alloc, beta in zip(bars, allocs, betas):
    ax.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{alloc:.0%} (beta={beta:.2f})",
        va="center",
        fontsize=10,
    )

ax.set_xlabel("Recommended Stock Allocation", fontsize=12)
ax.set_title("How Industry Affects Your Portfolio Allocation", fontsize=14, fontweight="bold")
ax.set_xlim(0, 1.15)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## Sensitivity Analysis: Beta vs. H/W Ratio

The impact of beta depends on how large your human capital is relative to financial wealth. When H/W is small (near retirement), beta barely matters. When H/W is large (early career), beta has a dramatic effect.

In [ ]:
import numpy as np

from lifecycle_allocation import risk_tolerance_to_gamma

betas = np.arange(0, 1.01, 0.05)
hw_ratios = [5, 10, 20]

gamma = risk_tolerance_to_gamma(6)
a_star_val, _, _ = alpha_star(market, gamma)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#2196F3", "#FF9800", "#E91E63"]

for hw, color in zip(hw_ratios, colors):
    allocs = []
    for beta in betas:
        raw = a_star_val * (1.0 + (1.0 - beta) * hw)
        clamped = max(0.0, min(raw, 1.0))
        allocs.append(clamped)
    ax.plot(betas, allocs, linewidth=2.5, label=f"H/W = {hw}", color=color)

ax.set_xlabel("Human Capital Beta", fontsize=12)
ax.set_ylabel("Recommended Stock Allocation", fontsize=12)
ax.set_title(
    "Allocation Sensitivity to Human Capital Beta",
    fontsize=14,
    fontweight="bold",
)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_xlim(0, 1.0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.axhline(
    y=a_star_val,
    color="gray",
    linewidth=1,
    linestyle=":",
    label=f"alpha* = {a_star_val:.1%}",
)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## Case Study 1: Young Tech Worker with RSUs

**Profile:** 28-year-old software engineer. Base salary \$180k, plus significant RSU grants. About 40% of total comp is equity-based, and their employer's stock is highly correlated with the broader market.

This person's income risk is meaningfully tied to equities, so we use `beta=0.6` (matching the `tech_with_rsus` calibration).

In [ ]:
tech_worker = InvestorProfile(
    age=28,
    retirement_age=67,
    investable_wealth=80_000,
    after_tax_income=180_000,
    risk_tolerance=7,
    human_capital_model=HumanCapitalSpec(beta=0.6, industry="tech_with_rsus"),
)

r_tech_worker = recommended_stock_share(tech_worker, market)

print(f"Recommended allocation: {r_tech_worker.alpha_recommended:.1%}")
print(f"Human capital: ${r_tech_worker.human_capital:,.0f}")
print(f"  Bond-like: ${r_tech_worker.components['human_capital_bond_like']:,.0f}")
print(f"  Equity-like: ${r_tech_worker.components['human_capital_equity_like']:,.0f}")
print()
print(r_tech_worker.explain)

---
## Case Study 2: Government Employee

**Profile:** 35-year-old federal employee. Stable salary of \$75k with a defined-benefit pension. Income is virtually uncorrelated with the stock market. This is the classic bond-like human capital scenario, so `beta=0`.

In [ ]:
gov_worker = InvestorProfile(
    age=35,
    retirement_age=67,
    investable_wealth=50_000,
    after_tax_income=75_000,
    risk_tolerance=5,
    human_capital_model=HumanCapitalSpec(beta=0.0, industry="government"),
)

r_gov = recommended_stock_share(gov_worker, market)

print(f"Recommended allocation: {r_gov.alpha_recommended:.1%}")
print(f"Human capital: ${r_gov.human_capital:,.0f}")
print(f"  Bond-like: ${r_gov.components['human_capital_bond_like']:,.0f}")
print(f"  Equity-like: ${r_gov.components['human_capital_equity_like']:,.0f}")
print()
print(r_gov.explain)

---
## Case Study 3: Startup Founder

**Profile:** 32-year-old startup founder. Modest salary of \$100k, but most of their net worth is tied up in private company equity. Their income, job security, and wealth are all highly correlated with market conditions and venture capital sentiment. We use `beta=0.75` (the `tech_startup` calibration).

In [ ]:
founder = InvestorProfile(
    age=32,
    retirement_age=67,
    investable_wealth=200_000,
    after_tax_income=100_000,
    risk_tolerance=6,
    human_capital_model=HumanCapitalSpec(beta=0.75, industry="tech_startup"),
)

r_founder = recommended_stock_share(founder, market)

print(f"Recommended allocation: {r_founder.alpha_recommended:.1%}")
print(f"Human capital: ${r_founder.human_capital:,.0f}")
print(f"  Bond-like: ${r_founder.components['human_capital_bond_like']:,.0f}")
print(f"  Equity-like: ${r_founder.components['human_capital_equity_like']:,.0f}")
print()
print(r_founder.explain)

---
## Glide Path Comparison: Beta = 0, 0.3, 0.6

How does the allocation evolve over a career for different levels of human capital risk? We simulate wealth growth at 5% per year with a 15% savings rate pre-retirement, then plot the allocation from age 25 to 80.

In [ ]:
ages = list(range(25, 81))
beta_values = [0.0, 0.3, 0.6]
beta_labels = ["beta=0 (bond-like)", "beta=0.3 (moderate)", "beta=0.6 (equity-like)"]
beta_colors = ["#4CAF50", "#FF9800", "#E91E63"]

fig, ax = plt.subplots(figsize=(10, 6))

for beta, label, color in zip(beta_values, beta_labels, beta_colors):
    allocs = []
    wealth = 50_000.0
    income = 100_000.0

    for age in ages:
        p = InvestorProfile(
            age=age,
            retirement_age=67,
            investable_wealth=max(wealth, 1_000),
            after_tax_income=income if age < 67 else None,
            risk_tolerance=6,
            human_capital_model=HumanCapitalSpec(beta=beta),
        )
        r = recommended_stock_share(p, market)
        allocs.append(r.alpha_recommended)

        # Simulate wealth growth: 5% return + 15% savings rate pre-retirement
        if age < 67:
            wealth = wealth * 1.05 + income * 0.15
        else:
            wealth = wealth * 1.05 - 40_000  # draw down in retirement

    ax.plot(ages, allocs, linewidth=2.5, label=label, color=color)

ax.set_xlabel("Age", fontsize=12)
ax.set_ylabel("Stock Allocation", fontsize=12)
ax.set_title("Lifecycle Glide Paths by Human Capital Beta", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.axvline(x=67, color="gray", linewidth=1, linestyle=":", alpha=0.7)
ax.text(67.5, 0.95, "Retirement", fontsize=10, color="gray")
plt.tight_layout()
plt.show()

---
## Practical Guidance: Estimating Your Own Beta

You do not need a precise number. A rough estimate is far better than ignoring the issue (which implicitly assumes beta=0). Here are some rules of thumb:

**Compensation structure:**
- If more than 30% of your total compensation is in equity, RSUs, or stock options, use **beta >= 0.4**.
- If you receive annual bonuses tied to company or division performance, add **0.05 to 0.15** to your base estimate.
- If you hold concentrated stock from your employer (e.g., vested RSUs you have not sold), your *effective* beta is even higher.

**Industry cyclicality:**
- If your industry is cyclical (tech, finance, real estate, oil and gas), add **0.1 to 0.2** to your base estimate.
- If your industry is defensive (government, utilities, healthcare, education), your base is likely **0.0 to 0.15**.

**Startup and entrepreneurial risk:**
- If you are a startup founder or early employee with significant equity, use **beta >= 0.6**.
- If your equity is in a pre-revenue company, consider **beta >= 0.8**.

**Quick reference from built-in calibrations:**

| Situation | Suggested Beta |
|---|---|
| Government, tenured academic | 0.00 |
| Healthcare, utilities | 0.10 |
| General private sector | 0.30 |
| Tech (salary only) | 0.40 |
| Tech with RSUs | 0.60 |
| Startup with equity | 0.75 |
| Equity-heavy startup founder | 0.85 |

You can also print the full table of built-in betas:

In [ ]:
print(f"{'Industry':<30} {'Beta':>6}")
print("-" * 38)
for industry, beta in sorted(INDUSTRY_BETAS.items(), key=lambda x: x[1]):
    print(f"{industry:<30} {beta:>5.2f}")

---
## Conclusion

The human capital beta feature captures a simple but important insight: **your job is part of your portfolio.** If your income moves with the stock market, treating it as a bond overstates the diversification benefit and leads to an overly aggressive equity allocation.

Key takeaways:
- Beta = 0 recovers the standard model. It is appropriate for workers with stable, market-independent income.
- Higher beta reduces the recommended equity allocation, sometimes substantially for young workers with high H/W ratios.
- The effect fades as you approach retirement, because human capital shrinks relative to financial wealth regardless of beta.
- You do not need a precise beta. Even a rough estimate ("am I closer to a government worker or a startup founder?") improves on the default assumption.

For the full theoretical foundation, see [Choi et al. (2024), "Popular Personal Financial Advice versus the Professors"](https://www.nber.org/papers/w34166).

> **Disclaimer:** This is for education and research purposes only. It is not investment advice. Your actual allocation should consider your full financial picture, tax situation, risk capacity, and goals.